# 02. raw 파일 SQLite 적재

이 노트북은 **raw 적재와 기본 스키마 설계**를 수행합니다.

## 목표

1. SQLite DB 파일을 생성한다.
2. raw 테이블 4개를 생성한다.
3. 원본 CSV/XLSX 파일을 raw 테이블에 적재한다.
4. 적재 후 행 수와 기간 범위를 확인한다.

## 생성되는 DB

```text
data/seoul_bike.db
```

## 생성되는 raw 테이블

```text
raw_signup_month
raw_ride_month
raw_station_snapshot
raw_station_usage_month
```

주의: 이 노트북은 기존 raw 테이블을 DROP 후 재생성합니다.


## 0. 라이브러리 불러오기

In [1]:
from pathlib import Path
from datetime import datetime
import re
import sqlite3

import pandas as pd
import numpy as np


## 1. 경로 설정

In [2]:
cwd = Path.cwd()

if cwd.name in ["notebooks", "scripts"]:
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
DB_PATH = DATA_DIR / "seoul_bike.db"
SQL_DIR = PROJECT_ROOT / "sql"

# ChatGPT 업로드 환경처럼 /mnt/data에 파일이 바로 있는 경우를 위한 fallback
if not RAW_DIR.exists() and Path("/mnt/data").exists():
    RAW_DIR = Path("/mnt/data")
    DATA_DIR = Path("/mnt/data")
    DB_PATH = DATA_DIR / "seoul_bike.db"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

print("원본 데이터 폴더:", RAW_DIR)
print("SQLite DB 경로:", DB_PATH)


원본 데이터 폴더: d:\seoul-bike-aarrr-analysis\data\raw
SQLite DB 경로: d:\seoul-bike-aarrr-analysis\data\seoul_bike.db


## 2. raw 테이블 생성 SQL

In [6]:
CREATE_RAW_SQL = """DROP TABLE IF EXISTS raw_signup_month;
DROP TABLE IF EXISTS raw_ride_month;
DROP TABLE IF EXISTS raw_station_snapshot;
DROP TABLE IF EXISTS raw_station_usage_month;

CREATE TABLE raw_signup_month (
    source_file TEXT,
    signup_ym INTEGER,
    member_type_raw TEXT,
    age_band_raw TEXT,
    gender_raw TEXT,
    signup_cnt_raw INTEGER,
    loaded_at TEXT
);

CREATE TABLE raw_ride_month (
    source_file TEXT,
    ride_ym INTEGER,
    station_id_raw TEXT,
    station_name_raw TEXT,
    pass_type_raw TEXT,
    gender_raw TEXT,
    age_band_raw TEXT,
    ride_cnt_raw INTEGER,
    exercise_amt_raw TEXT,
    carbon_amt_raw TEXT,
    distance_m_raw TEXT,
    duration_min_raw TEXT,
    loaded_at TEXT
);

CREATE TABLE raw_station_snapshot (
    snapshot_ym INTEGER,
    source_file TEXT,
    station_id_raw TEXT,
    station_name_raw TEXT,
    district_raw TEXT,
    address_raw TEXT,
    lat_raw TEXT,
    lon_raw TEXT,
    installed_at_raw TEXT,
    rack_lcd_raw TEXT,
    rack_qr_raw TEXT,
    operation_type_raw TEXT,
    loaded_at TEXT
);

CREATE TABLE raw_station_usage_month (
    source_file TEXT,
    usage_ym INTEGER,
    district_raw TEXT,
    station_name_raw TEXT,
    checkout_cnt_raw INTEGER,
    return_cnt_raw INTEGER,
    loaded_at TEXT
);

CREATE INDEX IF NOT EXISTS idx_raw_signup_month_ym ON raw_signup_month(signup_ym);
CREATE INDEX IF NOT EXISTS idx_raw_ride_month_ym ON raw_ride_month(ride_ym);
CREATE INDEX IF NOT EXISTS idx_raw_ride_station ON raw_ride_month(station_id_raw);
CREATE INDEX IF NOT EXISTS idx_raw_station_snapshot_ym ON raw_station_snapshot(snapshot_ym);
CREATE INDEX IF NOT EXISTS idx_raw_station_snapshot_station ON raw_station_snapshot(station_id_raw);
CREATE INDEX IF NOT EXISTS idx_raw_station_usage_month_ym ON raw_station_usage_month(usage_ym);
"""

(SQL_DIR / "01_create_raw_tables.sql").write_text(CREATE_RAW_SQL, encoding="utf-8")


1797

## 3. 파일 목록

In [3]:
def find_files(folder_name: str, pattern: str):
    folder = RAW_DIR / folder_name
    if folder.exists():
        return sorted(folder.glob(pattern))
    return sorted(RAW_DIR.glob(pattern))

files = {
    "signup_month": find_files("signup_month", "서울특별시 공공자전거 신규가입자 정보(월별)_*.csv"),
    "ride_month": find_files("ride_month", "서울특별시 공공자전거 이용정보(월별)_*.csv"),
    "station_snapshot": find_files("station_snapshot", "공공자전거 대여소 정보(*기준).xlsx"),
    "station_usage_month": find_files("station_usage_month", "서울특별시 공공자전거 대여소별 이용정보(월별)_*.csv"),
}

for key, paths in files.items():
    print(f"[{key}] {len(paths)}개")
    for p in paths:
        print(" -", p.name)


[signup_month] 5개
 - 서울특별시 공공자전거 신규가입자 정보(월별)_23.7-12.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_24.1-6.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_24.7-12.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_25.1-6.csv
 - 서울특별시 공공자전거 신규가입자 정보(월별)_25.7-12.csv
[ride_month] 5개
 - 서울특별시 공공자전거 이용정보(월별)_23.7-12.csv
 - 서울특별시 공공자전거 이용정보(월별)_24.1-6.csv
 - 서울특별시 공공자전거 이용정보(월별)_24.7-12.csv
 - 서울특별시 공공자전거 이용정보(월별)_25.1-6.csv
 - 서울특별시 공공자전거 이용정보(월별)_25.7-12.csv
[station_snapshot] 5개
 - 공공자전거 대여소 정보(23.12월 기준).xlsx
 - 공공자전거 대여소 정보(24.12월 기준).xlsx
 - 공공자전거 대여소 정보(24.6월 기준).xlsx
 - 공공자전거 대여소 정보(25.12월 기준).xlsx
 - 공공자전거 대여소 정보(25.6월 기준).xlsx
[station_usage_month] 5개
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_23.7-12.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_24.1-6.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_24.7-12.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_25.1-6.csv
 - 서울특별시 공공자전거 대여소별 이용정보(월별)_25.7-12.csv


## 4. 유틸 함수

In [4]:
encoding_candidates = ["utf-8-sig", "utf-8", "cp949", "euc-kr"]

def detect_csv_encoding(path: Path):
    for enc in encoding_candidates:
        try:
            pd.read_csv(path, encoding=enc, nrows=5)
            return enc
        except Exception:
            continue
    raise ValueError(f"인코딩 탐색 실패: {path}")

def now_str():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def to_int_series(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64")

def extract_snapshot_ym(file_name: str):
    m = re.search(r"\((\d{2})\.(\d{1,2})월 기준\)", file_name)
    if not m:
        return None
    yy, mm = m.groups()
    return int(f"20{yy}{int(mm):02d}")

def find_station_header_row(xlsx_path: Path, sheet_name="대여소현황", max_rows=20):
    preview = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=None, nrows=max_rows)
    header_keywords = ["대여소", "보관소", "자치구", "위도", "경도"]
    best_idx, best_score = None, -1

    for idx, row in preview.iterrows():
        values = " ".join([str(x) for x in row.values if pd.notna(x)])
        score = sum(kw in values for kw in header_keywords)
        if score > best_score:
            best_idx, best_score = idx, score
    return best_idx

def clean_col_name(x):
    return str(x).strip().replace("\n", " ")

def find_col(columns, candidates):
    cols = list(columns)
    normalized = {str(c).replace(" ", "").replace("\n", ""): c for c in cols}

    for cand in candidates:
        key = cand.replace(" ", "").replace("\n", "")
        if key in normalized:
            return normalized[key]

    for cand in candidates:
        key = cand.replace(" ", "").replace("\n", "")
        for norm, original in normalized.items():
            if key in norm or norm in key:
                return original
    return None


## 5. DB 생성 및 적재

In [8]:
conn = sqlite3.connect(DB_PATH)
conn.executescript(CREATE_RAW_SQL)
conn.commit()

def load_raw_signup_month(conn, paths):
    total = 0
    for path in paths:
        enc = detect_csv_encoding(path)
        df = pd.read_csv(path, encoding=enc)

        out = pd.DataFrame({
            "source_file": path.name,
            "signup_ym": to_int_series(df["가입년월"]),
            "member_type_raw": df["회원구분"].astype(str).str.strip(),
            "age_band_raw": df["연령대"].astype(str).str.strip(),
            "gender_raw": df["성별"].astype(str).str.strip(),
            "signup_cnt_raw": to_int_series(df["가입건수"]),
            "loaded_at": now_str()
        })

        out.to_sql("raw_signup_month", conn, if_exists="append", index=False)
        total += len(out)
        print(f"loaded signup: {path.name} / rows={len(out):,} / encoding={enc}")
    print(f"raw_signup_month 총 적재 행 수: {total:,}")

def load_raw_ride_month(conn, paths, chunksize=100_000):
    total = 0
    for path in paths:
        enc = detect_csv_encoding(path)
        file_total = 0

        for chunk in pd.read_csv(path, encoding=enc, chunksize=chunksize):
            out = pd.DataFrame({
                "source_file": path.name,
                "ride_ym": to_int_series(chunk["대여일자"]),
                "station_id_raw": chunk["대여소번호"].astype(str).str.strip(),
                "station_name_raw": chunk["대여소명"].astype(str).str.strip(),
                "pass_type_raw": chunk["대여구분코드"].astype(str).str.strip(),
                "gender_raw": chunk["성별"].astype(str).str.strip(),
                "age_band_raw": chunk["연령대코드"].astype(str).str.strip(),
                "ride_cnt_raw": to_int_series(chunk["이용건수"]),
                "exercise_amt_raw": chunk["운동량"].astype(str).str.strip(),
                "carbon_amt_raw": chunk["탄소량"].astype(str).str.strip(),
                "distance_m_raw": chunk["이동거리(M)"].astype(str).str.strip(),
                "duration_min_raw": chunk["이용시간(분)"].astype(str).str.strip(),
                "loaded_at": now_str()
            })

            out.to_sql("raw_ride_month", conn, if_exists="append", index=False)
            file_total += len(out)
            total += len(out)

        print(f"loaded ride: {path.name} / rows={file_total:,} / encoding={enc}")
    print(f"raw_ride_month 총 적재 행 수: {total:,}")

def load_raw_station_snapshot(conn, paths):
    total = 0

    station_cols = [
        "station_id_raw",
        "station_name_raw",
        "district_raw",
        "address_raw",
        "lat_raw",
        "lon_raw",
        "installed_at_raw",
        "rack_lcd_raw",
        "rack_qr_raw",
        "operation_type_raw"
    ]

    for path in paths:
        xl = pd.ExcelFile(path)
        sheet = "대여소현황" if "대여소현황" in xl.sheet_names else xl.sheet_names[0]

        df = pd.read_excel(
            path,
            sheet_name=sheet,
            header=None,
            skiprows=5,
            names=station_cols
        )

        df = df.dropna(how="all")

        # 대여소번호가 실제 숫자인 행만 남김
        df = df[pd.to_numeric(df["station_id_raw"], errors="coerce").notna()].copy()

        out = pd.DataFrame({
            "snapshot_ym": extract_snapshot_ym(path.name),
            "source_file": path.name,
            "station_id_raw": df["station_id_raw"].astype(str).str.strip(),
            "station_name_raw": df["station_name_raw"].astype(str).str.strip(),
            "district_raw": df["district_raw"].astype(str).str.strip(),
            "address_raw": df["address_raw"].astype(str).str.strip(),
            "lat_raw": df["lat_raw"].astype(str).str.strip(),
            "lon_raw": df["lon_raw"].astype(str).str.strip(),
            "installed_at_raw": df["installed_at_raw"].astype(str).str.strip(),
            "rack_lcd_raw": df["rack_lcd_raw"].astype(str).str.strip(),
            "rack_qr_raw": df["rack_qr_raw"].astype(str).str.strip(),
            "operation_type_raw": df["operation_type_raw"].astype(str).str.strip(),
            "loaded_at": now_str()
        })

        out.to_sql("raw_station_snapshot", conn, if_exists="append", index=False)
        total += len(out)

        print(
            f"loaded station snapshot: {path.name} "
            f"/ rows={len(out):,} "
            f"/ sheet={sheet} "
            f"/ snapshot_ym={extract_snapshot_ym(path.name)}"
        )

    print(f"raw_station_snapshot 총 적재 행 수: {total:,}")

def load_raw_station_usage_month(conn, paths):
    total = 0
    for path in paths:
        enc = detect_csv_encoding(path)
        df = pd.read_csv(path, encoding=enc)

        out = pd.DataFrame({
            "source_file": path.name,
            "usage_ym": to_int_series(df["기준년월"]),
            "district_raw": df["자치구"].astype(str).str.strip(),
            "station_name_raw": df["대여소명"].astype(str).str.strip(),
            "checkout_cnt_raw": to_int_series(df["대여건수"]),
            "return_cnt_raw": to_int_series(df["반납건수"]),
            "loaded_at": now_str()
        })

        out.to_sql("raw_station_usage_month", conn, if_exists="append", index=False)
        total += len(out)
        print(f"loaded station usage: {path.name} / rows={len(out):,} / encoding={enc}")
    print(f"raw_station_usage_month 총 적재 행 수: {total:,}")

conn.executescript(CREATE_RAW_SQL)
conn.commit()

load_raw_signup_month(conn, files["signup_month"])
load_raw_ride_month(conn, files["ride_month"])
load_raw_station_snapshot(conn, files["station_snapshot"])
load_raw_station_usage_month(conn, files["station_usage_month"])


loaded signup: 서울특별시 공공자전거 신규가입자 정보(월별)_23.7-12.csv / rows=96 / encoding=cp949
loaded signup: 서울특별시 공공자전거 신규가입자 정보(월별)_24.1-6.csv / rows=96 / encoding=cp949
loaded signup: 서울특별시 공공자전거 신규가입자 정보(월별)_24.7-12.csv / rows=96 / encoding=cp949
loaded signup: 서울특별시 공공자전거 신규가입자 정보(월별)_25.1-6.csv / rows=97 / encoding=cp949
loaded signup: 서울특별시 공공자전거 신규가입자 정보(월별)_25.7-12.csv / rows=96 / encoding=cp949
raw_signup_month 총 적재 행 수: 481
loaded ride: 서울특별시 공공자전거 이용정보(월별)_23.7-12.csv / rows=624,320 / encoding=cp949
loaded ride: 서울특별시 공공자전거 이용정보(월별)_24.1-6.csv / rows=613,732 / encoding=cp949
loaded ride: 서울특별시 공공자전거 이용정보(월별)_24.7-12.csv / rows=619,664 / encoding=cp949
loaded ride: 서울특별시 공공자전거 이용정보(월별)_25.1-6.csv / rows=600,437 / encoding=cp949
loaded ride: 서울특별시 공공자전거 이용정보(월별)_25.7-12.csv / rows=629,975 / encoding=cp949
raw_ride_month 총 적재 행 수: 3,088,128
loaded station snapshot: 공공자전거 대여소 정보(23.12월 기준).xlsx / rows=2,762 / sheet=대여소현황 / snapshot_ym=202312
loaded station snapshot: 공공자전거 대여소 정보(24.12월 기준).xl

## 6. 적재 결과 확인

In [9]:
raw_tables = [
    "raw_signup_month",
    "raw_ride_month",
    "raw_station_snapshot",
    "raw_station_usage_month"
]

print("\n[테이블별 행 수]")
for table in raw_tables:
    cnt = pd.read_sql_query(f"SELECT COUNT(*) AS row_count FROM {table}", conn).iloc[0, 0]
    print(table, f"{cnt:,}")

print("\n[기간 커버리지]")
queries = {
    "raw_signup_month": "SELECT MIN(signup_ym) AS min_ym, MAX(signup_ym) AS max_ym, COUNT(DISTINCT signup_ym) AS cnt FROM raw_signup_month",
    "raw_ride_month": "SELECT MIN(ride_ym) AS min_ym, MAX(ride_ym) AS max_ym, COUNT(DISTINCT ride_ym) AS cnt FROM raw_ride_month",
    "raw_station_snapshot": "SELECT MIN(snapshot_ym) AS min_ym, MAX(snapshot_ym) AS max_ym, COUNT(DISTINCT snapshot_ym) AS cnt FROM raw_station_snapshot",
    "raw_station_usage_month": "SELECT MIN(usage_ym) AS min_ym, MAX(usage_ym) AS max_ym, COUNT(DISTINCT usage_ym) AS cnt FROM raw_station_usage_month",
}
for table, q in queries.items():
    print(table)
    print(pd.read_sql_query(q, conn))

conn.close()
print("\n완료:", DB_PATH)



[테이블별 행 수]
raw_signup_month 481
raw_ride_month 3,088,128
raw_station_snapshot 13,870
raw_station_usage_month 82,142

[기간 커버리지]
raw_signup_month
   min_ym  max_ym  cnt
0  202307  202512   30
raw_ride_month
   min_ym  max_ym  cnt
0  202307  202512   30
raw_station_snapshot
   min_ym  max_ym  cnt
0  202312  202512    5
raw_station_usage_month
   min_ym  max_ym  cnt
0  202307  202512   30

완료: d:\seoul-bike-aarrr-analysis\data\seoul_bike.db


## 완료 기준

아래 조건을 만족하면 완료입니다.

- `data/seoul_bike.db` 파일이 생성되었다.
- raw 테이블 4개가 생성되었다.
- 원본 파일이 각 raw 테이블에 적재되었다.
- 이용정보 raw 테이블에 `2023.10~2023.12` 계산 보조 데이터가 포함되었다.
- 대여소 스냅샷 raw 테이블에 `2023.12`, `2024.06`, `2024.12`, `2025.06`, `2025.12` 기준월이 포함되었다.
- 테이블별 row count를 확인했다.
- 월별 데이터의 기간 범위를 확인했다.
- 대여소 정보의 `snapshot_ym`이 정상적으로 들어갔다.
